# Level 2 — Task 2: SQL for Business Analytics
**Internship:** Codveda Technology — Business Analytics  
**Objective:** Use SQL to query and manipulate business data — covering basic and advanced queries, aggregations, joins, subqueries, and query optimization.  
**Approach:** We load the churn dataset into a **SQLite in-memory database** — no server required. Every query runs natively and the notebook is fully portable.

---

## 0. Setup & Imports

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded ✓')

---
## 1. Database Setup — Load CSVs into SQLite
We create three tables that simulate a real business database:
- **`customers`** — core account and plan info
- **`usage`** — all usage metrics (day, evening, night, international)
- **`support`** — customer service call records

> **Reusability tip:** Replace the CSV paths to point to your own data. The `build_database()` function accepts any DataFrame.

In [ ]:
DATA_DIR = '../data'

# Load cleaned churn data
churn = pd.read_csv(os.path.join(DATA_DIR, 'churn_cleaned.csv'))
churn['customer_id'] = range(1, len(churn) + 1)  # Add surrogate key

def build_database(df):
    """
    Reusable: creates an in-memory SQLite database from a churn DataFrame.
    Splits into normalised tables: customers, usage, support.
    Returns the connection object.
    """
    conn = sqlite3.connect(':memory:')

    # ── Table 1: customers ──────────────────────────────────────────────────
    customers = df[[
        'customer_id', 'State', 'Account length', 'Area code',
        'International plan', 'Voice mail plan',
        'Number vmail messages', 'Churn'
    ]].copy()
    customers.columns = [
        'customer_id', 'state', 'account_length', 'area_code',
        'intl_plan', 'voicemail_plan',
        'vmail_messages', 'churn'
    ]
    customers.to_sql('customers', conn, index=False, if_exists='replace')

    # ── Table 2: usage ──────────────────────────────────────────────────────
    usage = df[[
        'customer_id',
        'Total day minutes',   'Total day calls',   'Total day charge',
        'Total eve minutes',   'Total eve calls',   'Total eve charge',
        'Total night minutes', 'Total night calls', 'Total night charge',
        'Total intl minutes',  'Total intl calls',  'Total intl charge'
    ]].copy()
    usage.columns = [
        'customer_id',
        'day_minutes',   'day_calls',   'day_charge',
        'eve_minutes',   'eve_calls',   'eve_charge',
        'night_minutes', 'night_calls', 'night_charge',
        'intl_minutes',  'intl_calls',  'intl_charge'
    ]
    usage.to_sql('usage', conn, index=False, if_exists='replace')

    # ── Table 3: support ────────────────────────────────────────────────────
    support = df[['customer_id', 'Customer service calls']].copy()
    support.columns = ['customer_id', 'service_calls']
    support.to_sql('support', conn, index=False, if_exists='replace')

    return conn

conn = build_database(churn)

# Verify tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print('Database tables created:')
for t in tables['name']:
    count = pd.read_sql(f'SELECT COUNT(*) AS rows FROM {t}', conn).iloc[0,0]
    print(f'  {t:12s} → {count:,} rows')

---
## 2. Query Helper
A reusable function to run any SQL and display results cleanly.

In [ ]:
def run_query(conn, sql, title=''):
    """
    Reusable: executes a SQL query and displays results as a DataFrame.
    conn  — SQLite connection
    sql   — SQL string
    title — optional label printed above the result
    """
    if title:
        print(f'── {title} ──')
    df = pd.read_sql(sql, conn)
    display(df)
    return df

print('run_query() helper ready ✓')

---
## 3. Basic SQL Queries
### 3.1 SELECT, WHERE, ORDER BY, LIMIT

In [ ]:
run_query(conn, """
    SELECT customer_id, state, account_length, intl_plan, churn
    FROM   customers
    WHERE  churn = 1
    ORDER  BY account_length DESC
    LIMIT  10
""", 'Top 10 Churned Customers by Account Length')

In [ ]:
run_query(conn, """
    SELECT customer_id, day_minutes, day_charge, intl_minutes, intl_charge
    FROM   usage
    WHERE  day_minutes > 250
      AND  intl_charge > 3.0
    ORDER  BY day_charge DESC
    LIMIT  10
""", 'High Usage Customers (Day > 250 min AND Intl Charge > $3)')

---
## 4. Data Aggregation — SUM, AVG, COUNT, GROUP BY
### 4.1 Churn summary statistics

In [ ]:
run_query(conn, """
    SELECT
        CASE WHEN churn = 1 THEN 'Churned' ELSE 'Retained' END AS status,
        COUNT(*)                          AS total_customers,
        ROUND(COUNT(*) * 100.0 /
              SUM(COUNT(*)) OVER(), 1)    AS pct_of_total,
        ROUND(AVG(account_length), 1)     AS avg_account_length,
        ROUND(AVG(vmail_messages), 1)     AS avg_vmail_msgs
    FROM customers
    GROUP BY churn
    ORDER BY churn DESC
""", 'Churn Summary — COUNT, AVG, Window Function')

In [ ]:
run_query(conn, """
    SELECT
        CASE WHEN c.churn = 1 THEN 'Churned' ELSE 'Retained' END AS status,
        ROUND(AVG(u.day_minutes),   1)  AS avg_day_mins,
        ROUND(AVG(u.eve_minutes),   1)  AS avg_eve_mins,
        ROUND(AVG(u.night_minutes), 1)  AS avg_night_mins,
        ROUND(AVG(u.intl_minutes),  1)  AS avg_intl_mins,
        ROUND(SUM(u.day_charge + u.eve_charge +
                  u.night_charge + u.intl_charge)
              / COUNT(*), 2)            AS avg_total_charge
    FROM   customers c
    JOIN   usage u ON c.customer_id = u.customer_id
    GROUP  BY c.churn
""", 'Average Usage Metrics by Churn Status')

### 4.2 Aggregation by State — Top churning regions

In [ ]:
state_churn = run_query(conn, """
    SELECT
        state,
        COUNT(*)                                      AS total_customers,
        SUM(churn)                                    AS churned,
        ROUND(SUM(churn) * 100.0 / COUNT(*), 1)       AS churn_rate_pct
    FROM   customers
    GROUP  BY state
    HAVING total_customers >= 30
    ORDER  BY churn_rate_pct DESC
    LIMIT  15
""", 'Top 15 States by Churn Rate (min 30 customers)')

# Visualise
fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(state_churn['state'], state_churn['churn_rate_pct'],
              color='#e74c3c', edgecolor='white', alpha=0.85)
ax.axhline(y=state_churn['churn_rate_pct'].mean(), color='#2c3e50',
           linestyle='--', linewidth=1.5, label='Avg churn rate')
ax.set_title('Top 15 States by Churn Rate', fontweight='bold', fontsize=13)
ax.set_xlabel('State')
ax.set_ylabel('Churn Rate (%)')
ax.legend()
for bar, val in zip(bars, state_churn['churn_rate_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.0f}%', ha='center', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.savefig('churn_by_state.png', bbox_inches='tight')
plt.show()

---
## 5. Joins — Combining Multiple Tables
### 5.1 INNER JOIN — customers + usage + support

In [ ]:
run_query(conn, """
    SELECT
        c.customer_id,
        c.state,
        c.intl_plan,
        c.churn,
        ROUND(u.day_charge + u.eve_charge +
              u.night_charge + u.intl_charge, 2)  AS total_charge,
        s.service_calls
    FROM   customers c
    INNER  JOIN usage   u ON c.customer_id = u.customer_id
    INNER  JOIN support s ON c.customer_id = s.customer_id
    WHERE  c.churn = 1
    ORDER  BY total_charge DESC
    LIMIT  12
""", 'INNER JOIN — Churned Customers: Full Profile')

In [ ]:
service_analysis = run_query(conn, """
    SELECT
        s.service_calls,
        COUNT(*)                                    AS total_customers,
        SUM(c.churn)                                AS churned,
        ROUND(SUM(c.churn)*100.0 / COUNT(*), 1)     AS churn_rate_pct,
        ROUND(AVG(u.day_minutes), 1)                AS avg_day_minutes,
        ROUND(AVG(u.day_charge + u.eve_charge +
                  u.night_charge + u.intl_charge),2) AS avg_total_charge
    FROM   support s
    JOIN   customers c ON s.customer_id = c.customer_id
    JOIN   usage     u ON s.customer_id = u.customer_id
    GROUP  BY s.service_calls
    ORDER  BY s.service_calls
""", 'Three-Table JOIN — Churn Rate by Service Call Volume')

# Visualise
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
bars = ax1.bar(service_analysis['service_calls'],
               service_analysis['total_customers'],
               color='#3498db', edgecolor='white', alpha=0.7, label='Total Customers')
ax2.plot(service_analysis['service_calls'],
         service_analysis['churn_rate_pct'],
         color='#e74c3c', marker='o', linewidth=2.5,
         markersize=8, label='Churn Rate %')
ax1.set_xlabel('Number of Service Calls', fontweight='bold')
ax1.set_ylabel('Total Customers', color='#3498db')
ax2.set_ylabel('Churn Rate (%)', color='#e74c3c')
ax1.set_title('Service Call Volume vs Churn Rate\n(Three-Table SQL Join)',
              fontweight='bold', fontsize=13)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig('service_calls_churn.png', bbox_inches='tight')
plt.show()

---
## 6. Advanced Queries
### 6.1 Subquery — Customers Above Average Charge

In [ ]:
run_query(conn, """
    SELECT
        c.customer_id,
        c.state,
        c.churn,
        ROUND(u.day_charge + u.eve_charge +
              u.night_charge + u.intl_charge, 2) AS total_charge
    FROM customers c
    JOIN usage u ON c.customer_id = u.customer_id
    WHERE (u.day_charge + u.eve_charge + u.night_charge + u.intl_charge) >
          (
              SELECT AVG(day_charge + eve_charge + night_charge + intl_charge)
              FROM   usage
          )
    ORDER BY total_charge DESC
    LIMIT 10
""", 'Subquery — Customers with Above-Average Total Charge')

### 6.2 CTE (Common Table Expression) — High-Risk Customer Segments

In [ ]:
run_query(conn, """
    WITH customer_risk AS (
        SELECT
            c.customer_id,
            c.state,
            c.intl_plan,
            c.churn,
            s.service_calls,
            ROUND(u.day_charge + u.eve_charge +
                  u.night_charge + u.intl_charge, 2) AS total_charge,
            CASE
                WHEN s.service_calls >= 4 AND c.intl_plan = 1
                    THEN 'Very High Risk'
                WHEN s.service_calls >= 4 OR c.intl_plan = 1
                    THEN 'High Risk'
                WHEN s.service_calls >= 2
                    THEN 'Medium Risk'
                ELSE 'Low Risk'
            END AS risk_segment
        FROM customers c
        JOIN usage   u ON c.customer_id = u.customer_id
        JOIN support s ON c.customer_id = s.customer_id
    )
    SELECT
        risk_segment,
        COUNT(*)                                   AS customers,
        SUM(churn)                                 AS churned,
        ROUND(SUM(churn)*100.0/COUNT(*), 1)        AS churn_rate_pct,
        ROUND(AVG(total_charge), 2)                AS avg_charge,
        ROUND(AVG(service_calls), 1)               AS avg_service_calls
    FROM customer_risk
    GROUP BY risk_segment
    ORDER BY churn_rate_pct DESC
""", 'CTE — Customer Risk Segmentation')

### 6.3 Window Functions — Running Totals & Rankings

In [ ]:
run_query(conn, """
    SELECT
        state,
        total_customers,
        churned,
        churn_rate_pct,
        RANK() OVER (ORDER BY churn_rate_pct DESC) AS churn_rank,
        SUM(churned) OVER (ORDER BY churn_rate_pct DESC
                          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                          ) AS running_churned_total
    FROM (
        SELECT
            state,
            COUNT(*)                             AS total_customers,
            SUM(churn)                           AS churned,
            ROUND(SUM(churn)*100.0/COUNT(*), 1)  AS churn_rate_pct
        FROM customers
        GROUP BY state
        HAVING total_customers >= 30
    )
    ORDER BY churn_rank
    LIMIT 12
""", 'Window Functions — State Churn Rank + Running Total')

---
## 7. Query Optimization
### 7.1 Using EXPLAIN QUERY PLAN to inspect execution

In [ ]:
# Check execution plan BEFORE index
plan_before = pd.read_sql("""
    EXPLAIN QUERY PLAN
    SELECT c.customer_id, c.churn, u.day_minutes
    FROM customers c
    JOIN usage u ON c.customer_id = u.customer_id
    WHERE c.churn = 1
""", conn)
print('Query plan BEFORE index:')
display(plan_before)

In [ ]:
# Create indexes on frequently queried columns
conn.execute('CREATE INDEX IF NOT EXISTS idx_customers_churn   ON customers(churn)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_customers_state   ON customers(state)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_usage_customer    ON usage(customer_id)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_support_calls     ON support(service_calls)')
conn.commit()

# Check execution plan AFTER index
plan_after = pd.read_sql("""
    EXPLAIN QUERY PLAN
    SELECT c.customer_id, c.churn, u.day_minutes
    FROM customers c
    JOIN usage u ON c.customer_id = u.customer_id
    WHERE c.churn = 1
""", conn)
print('Query plan AFTER index:')
display(plan_after)
print('\n✓ Index scan replaces full table scan — query is now optimized')

### 7.2 Timing benchmark — index vs no-index equivalent

In [ ]:
import time

def benchmark_query(conn, sql, runs=50, label='Query'):
    """Reusable: times average execution of a SQL query over N runs."""
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        pd.read_sql(sql, conn)
        times.append(time.perf_counter() - start)
    avg_ms = np.mean(times) * 1000
    print(f'  {label}: avg {avg_ms:.3f} ms over {runs} runs')
    return avg_ms

sql_churn_filter = """
    SELECT c.customer_id, c.churn, u.day_minutes
    FROM   customers c
    JOIN   usage u ON c.customer_id = u.customer_id
    WHERE  c.churn = 1
"""

sql_full_scan = """
    SELECT customer_id, account_length, state
    FROM   customers
    WHERE  account_length BETWEEN 50 AND 150
"""

print('── Query Benchmarks ──')
t1 = benchmark_query(conn, sql_churn_filter, label='Indexed churn JOIN')
t2 = benchmark_query(conn, sql_full_scan,    label='Range scan (no index)')

print(f'\n  Optimization tip: adding an index on `account_length`')
print(f'  would convert the range scan to an index range scan,\n  reducing I/O for large tables.')

---
## 8. Business Dashboard Query — Executive KPI Summary

In [ ]:
kpi = run_query(conn, """
    SELECT
        COUNT(*)                                          AS total_customers,
        SUM(c.churn)                                      AS total_churned,
        ROUND(SUM(c.churn)*100.0 / COUNT(*), 2)           AS churn_rate_pct,
        ROUND(AVG(u.day_charge + u.eve_charge +
                  u.night_charge + u.intl_charge), 2)     AS avg_monthly_charge,
        ROUND(SUM(u.day_charge + u.eve_charge +
                  u.night_charge + u.intl_charge), 2)     AS total_revenue,
        ROUND(SUM(CASE WHEN c.churn=1
                  THEN u.day_charge + u.eve_charge +
                       u.night_charge + u.intl_charge
                  ELSE 0 END), 2)                         AS revenue_lost_to_churn,
        ROUND(AVG(s.service_calls), 2)                    AS avg_service_calls,
        SUM(CASE WHEN c.intl_plan=1 THEN 1 ELSE 0 END)   AS intl_plan_customers
    FROM customers c
    JOIN usage   u ON c.customer_id = u.customer_id
    JOIN support s ON c.customer_id = s.customer_id
""", 'Executive KPI Dashboard — Single SQL Query')

# Pretty print as KPI cards
print('\n' + '━'*55)
print('  📊 EXECUTIVE KPI SUMMARY')
print('━'*55)
for col in kpi.columns:
    val = kpi.iloc[0][col]
    if 'revenue' in col or 'charge' in col:
        print(f'  {col:<30}: ${val:>12,.2f}')
    elif 'pct' in col or 'rate' in col:
        print(f'  {col:<30}: {val:>11.2f}%')
    else:
        print(f'  {col:<30}: {val:>12,.2f}')
print('━'*55)

In [ ]:
# Revenue breakdown chart
total_rev  = kpi.iloc[0]['total_revenue']
lost_rev   = kpi.iloc[0]['revenue_lost_to_churn']
kept_rev   = total_rev - lost_rev

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Donut chart — revenue retained vs lost
wedges, texts, autotexts = axes[0].pie(
    [kept_rev, lost_rev],
    labels=['Revenue Retained', 'Revenue Lost to Churn'],
    colors=['#2ecc71', '#e74c3c'],
    autopct='%1.1f%%', startangle=90,
    pctdistance=0.82,
    wedgeprops=dict(width=0.55))
axes[0].set_title('Revenue: Retained vs Lost to Churn', fontweight='bold')

# Plan distribution
plan_data = pd.read_sql("""
    SELECT
        CASE WHEN intl_plan=1 THEN 'Intl Plan' ELSE 'No Intl Plan' END AS plan_type,
        CASE WHEN voicemail_plan=1 THEN 'Voicemail' ELSE 'No Voicemail' END AS vm_type,
        COUNT(*) AS count
    FROM customers
    GROUP BY intl_plan, voicemail_plan
""", conn)
labels = [f"{r['plan_type']}\n{r['vm_type']}" for _, r in plan_data.iterrows()]
axes[1].bar(labels, plan_data['count'],
            color=['#3498db','#9b59b6','#e74c3c','#2ecc71'],
            edgecolor='white')
axes[1].set_title('Customer Plan Distribution', fontweight='bold')
axes[1].set_ylabel('Number of Customers')
for i, v in enumerate(plan_data['count']):
    axes[1].text(i, v + 20, f'{v:,}', ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Business Intelligence Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('bi_dashboard.png', bbox_inches='tight')
plt.show()

---
## 9. SQL Skills Summary

| SQL Concept | Applied In |
|-------------|------------|
| SELECT, WHERE, ORDER BY, LIMIT | Basic customer filtering |
| COUNT, SUM, AVG, GROUP BY | Churn & usage aggregations |
| HAVING | State-level filtering (min 30 customers) |
| INNER JOIN (2 & 3 tables) | Cross-table customer profiling |
| Subquery | Above-average charge detection |
| CTE (WITH clause) | Risk segmentation |
| Window Functions (RANK, SUM OVER) | State ranking & running totals |
| EXPLAIN QUERY PLAN | Execution plan inspection |
| CREATE INDEX | Query optimization |
| CASE WHEN | KPI computation & segmentation |

> **Business Impact:** The CTE risk segmentation query alone can drive targeted retention campaigns — customers in the *Very High Risk* segment should be prioritised for proactive outreach before they churn.